# 03 – Modeling

This notebook follows the CRISP-DM modeling phase.

The purpose is to develop predictive models for short-term utilization patterns of volumetric pumps based on movement tracking data.

As direct clinical utilization cannot be observed in the dataset, modelling focuses on a proxy outcome derived from movement behaviour.

Specifically, the target variable is defined as whether a volumetric pump registers a new movement event within a predefined future time window.

### Import libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

## Load prepared data

The modelling phase uses the processed dataset created during Data Preparation.

In [ ]:
df = pd.read_csv("../data_processed/transit_2026_prepared.csv")

df.head()


## 4.1 Select modeling technique

#### Modeling objective

The goal is to predict short-term movement activity as a proxy for utilization.

##### Target variable
Will a device move again within the next 4 hours?

This is formulated as a binary classification problem:

- 1 = yes, next movement occurs within 4 hours  
- 0 = no, device remains stationary longer  

#### Modeling techniques selected

**Logistic Regression**
Used as an interpretable baseline classification model.

**Random Forest Classifier**
Used to capture nonlinear relationships and interactions between predictors.

#### Modeling assumptions

The modelling approach relies on several assumptions:

- Movement events serve as a proxy for actual utilization  
- Historical movement behaviour contains predictive signal  
- Observations are temporally ordered and independent across devices  
- Missing location data does not systematically bias results  
- The selected features sufficiently capture relevant behavioural patterns 

### Construct timestamp for temporal ordering

To predict future movement, observations must be sorted chronologically for each device.

Although date and time were prepared previously, they are combined here into one timestamp variable for target construction.

In [ ]:
df["timestamp"] = pd.to_datetime(
    df["sd_date"].astype(str) + " " +
    df["st_time"].astype(str)
)

### Build target variable

The target captures whether the same device records another movement event within the next 4 hours.

In [ ]:
# Sort observations by device and time
df = df.sort_values(["object_key", "timestamp"])

# Find next movement timestamp for each device
df["next_time"] = (
    df.groupby("object_key")["timestamp"]
    .shift(-1)
)

# Calculate time until next movement in hours
df["time_to_next_hours"] = (
    (df["next_time"] - df["timestamp"])
    .dt.total_seconds() / 3600
)

# Binary target variable
df["target_move_8h"] = (
    df["time_to_next_hours"] <= 8
).astype(int)

df[[
    "object_key",
    "timestamp",
    "next_time",
    "time_to_next_hours",
    "target_move_8h"
]].head()

## 4.2 Generate test design

### Feature selection

Predictor variables are selected based on movement behaviour, temporal patterns, and spatial context identified in earlier analyses.

In [ ]:
features = [
    "distance_mtr",
    "duration_sec",
    "stay_duration_sec",
    "segment_count",
    "distance_floor",
    "hour",
    "time_period",
    "movement_ratio",
    "avg_speed",
    "activity_level",
    "sl_floor_local_name"
]

X = df[features]
y = df["target_move_8h"]

### Train-val-test split

The dataset is divided into training and test sets.

A time-based split is used to preserve class balance.

In [ ]:
# Sort by time
df = df.sort_values("timestamp")

# Define split indices (60% train, 20% validation, 20% test)
n = len(df)

train_end = int(n * 0.6)
val_end = int(n * 0.8)

# Create splits
train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]

# Define X and y
X_train = train_df[features]
y_train = train_df["target_move_8h"]

X_val = val_df[features]
y_val = val_df["target_move_8h"]

X_test = test_df[features]
y_test = test_df["target_move_8h"]

# Print shapes
print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

### Preprocessing
Categorical variables are one-hot encoded before modelling.

In [ ]:
categorical_features = [
    "time_period",
    "activity_level",
    "sl_floor_local_name"
]

numerical_features = [
    col for col in features
    if col not in categorical_features
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            "passthrough",
            numerical_features
        )
    ]
)

### Handle missing values before modelling

The error occurs because some predictor variables still contain missing values (`NaN`).

This is expected, especially for:

- `sl_floor_local_name`
- potentially derived variables affected by missing input  

Logistic Regression cannot train with missing values directly.

### Solution

Add imputation to the preprocessing pipeline:

- fill missing categorical values with `"Unknown"`
- fill missing numerical values with median values  

This preserves observations while making the dataset compatible with machine learning models.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Categorical preprocessing
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Numerical preprocessing
from sklearn.preprocessing import StandardScaler

# Numerical preprocessing
numerical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
# Combined preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            categorical_transformer,
            categorical_features
        ),
        (
            "num",
            numerical_transformer,
            numerical_features
        )
    ]
)

## 4.3 Build model

### Model 1: Logistic Regression

In [ ]:
log_model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(max_iter=3000))
])

log_model.fit(X_train, y_train)

#### Generate predictions

In [ ]:
y_pred_log = log_model.predict(X_val)
y_prob_log = log_model.predict_proba(X_val)[:, 1]

### Model 2: Random Forest

In [ ]:
rf_model = Pipeline([
    ("preprocessing", preprocessor),
    (
        "classifier",
        RandomForestClassifier(
            n_estimators=200,
            random_state=42
        )
    )
])

rf_model.fit(X_train, y_train)

#### Generate predictions

In [ ]:
y_pred_rf = rf_model.predict(X_val)
y_prob_rf = rf_model.predict_proba(X_val)[:, 1]

## 4.4 Assess model

### Evaluation function

In [ ]:
def evaluate_model(y_true, y_pred, y_prob, model_name):
    print(f"\n--- {model_name} ---")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall:", recall_score(y_true, y_pred))
    print("F1-score:", f1_score(y_true, y_pred))
    print("ROC-AUC:", roc_auc_score(y_true, y_prob))

### Asses Logistic Regression 

In [ ]:
evaluate_model(
    y_val,
    y_pred_log,
    y_prob_log,
    "Logistic Regression"
)

### Asses Random Forest

In [ ]:
evaluate_model(
    y_val,
    y_pred_rf,
    y_prob_rf,
    "Random Forest"
)

### Model assessment

Both models show predictive ability for short-term movement behaviour.

#### Logistic Regression
- Accuracy: 0.668  
- F1-score: 0.755  
- ROC-AUC: 0.701  

Provides a useful baseline but captures patterns less effectively.

#### Random Forest
- Accuracy: 0.725  
- F1-score: 0.789  
- ROC-AUC: 0.772  

Performs better across all metrics.

**Random Forest is the best-performing model.**

This suggests that historical movement behaviour contains useful predictive information for short-term utilization patterns.

### Model 3: XGBoost

XGBoost is included as an additional classification model to test whether gradient boosting improves predictive performance beyond Random Forest.

#### Model description

The final model is an XGBoost classifier based on gradient boosted decision trees.

The model builds an ensemble of sequential decision trees, where each tree corrects errors made by previous trees.

This allows the model to capture nonlinear relationships and interactions between movement behaviour, temporal patterns, and spatial context.

As a result, the model is well-suited for predicting short-term movement activity based on complex operational patterns.

In [ ]:
!pip install xgboost

In [ ]:
from xgboost import XGBClassifier

In [ ]:
xgb_model = Pipeline([
    ("preprocessing", preprocessor),
    (
        "classifier",
        XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            random_state=42,
            eval_metric="logloss"
        )
    )
])

xgb_model.fit(X_train, y_train)

### Generate predictions

In [ ]:
y_pred_xgb = xgb_model.predict(X_val)
y_prob_xgb = xgb_model.predict_proba(X_val)[:, 1]

### Assess XGBoost

In [ ]:
evaluate_model(
    y_val,
    y_pred_xgb,
    y_prob_xgb,
    "XGBoost"
)

### Model comparison

Three models were evaluated:

- Logistic Regression  
- Random Forest  
- XGBoost  

XGBoost achieved the best overall performance across evaluation metrics, particularly in terms of ROC-AUC and F1-score.

It was therefore selected as the final model.

## 4.5 Feature importance analysis


To better interpret the best-performing model, feature importance is examined for XGBoost.

This identifies which movement, temporal, and spatial variables contribute most to predicting short-term movement activity.

In [ ]:
import matplotlib.pyplot as plt

# Extract trained XGBoost classifier
model = xgb_model.named_steps["classifier"]

# Get transformed feature names after preprocessing
feature_names = (
    xgb_model.named_steps["preprocessing"]
    .get_feature_names_out()
)

# Create feature importance dataframe
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": model.feature_importances_
})

# Sort descending
importance_df = importance_df.sort_values(
    "importance",
    ascending=False
)

# Show top features
importance_df.head(10)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Extract trained XGBoost classifier
model = xgb_model.named_steps["classifier"]

# Get transformed feature names
feature_names = (
    xgb_model.named_steps["preprocessing"]
    .get_feature_names_out()
)

# Create dataframe
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": model.feature_importances_
})

# Clean feature names
importance_df["feature"] = importance_df["feature"].str.replace("cat__", "")
importance_df["feature"] = importance_df["feature"].str.replace("num__", "")

# Sort and select top 10
top_features = (
    importance_df
    .sort_values("importance", ascending=False)
    .head(10)
    .sort_values("importance")
)

# Create colors from viridis colormap
colors = plt.cm.viridis(
    np.linspace(0, 1, len(top_features))
)

# Plot
plt.figure(figsize=(8, 5))

plt.barh(
    top_features["feature"],
    top_features["importance"],
    color=colors
)

plt.title("Top 10 Feature Importances - XGBoost")
plt.xlabel("Importance")
plt.ylabel("Feature")

plt.tight_layout()
plt.show()

### Feature importance results

The XGBoost model indicates that both temporal patterns, device activity history, and location context are important for predicting short-term movement.

#### Most important predictors

**1. `time_period_day`**
Daytime activity is the strongest predictor.

This aligns with earlier descriptive findings showing that movement events are concentrated during daytime hours.

**2. `activity_level_high`**
Devices with historically high movement frequency are more likely to move again soon.

This confirms persistence in utilization behaviour across pumps.

**3. Floor-level location variables**
Several specific floors appear among the strongest predictors.

Examples include:
- Building L Outpatient  
- Building F F.01  
- Building E E.01  
- KE.01  

This suggests that local department context strongly influences movement behaviour.

**4. `stay_duration_sec`**
Longer stationary periods affect likelihood of future movement.

This reflects the importance of inactivity patterns.

**5. `movement_ratio`**
Devices spending relatively more time moving are more likely to continue showing short-term activity.

#### Overall interpretation

Prediction of short-term utilization is driven by a combination of:

- temporal demand patterns  
- historical device activity  
- departmental location  
- recent inactivity duration  

These findings support the idea that movement tracking data captures meaningful operational behaviour related to equipment utilization.

### Confusion matrix

The confusion matrix provides a detailed overview of correct and incorrect classifications made by the best-performing model.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_val, y_pred_xgb)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()

plt.title("Confusion Matrix - XGBoost")
plt.show()

The XGBoost model correctly identifies many movement events within the next 8 hours.

#### Correct classifications
- **True negatives:** 1,455 
  Correctly predicted no movement within 8 hours  

- **True positives:** 3,763
  Correctly predicted movement within 8 hours  

#### Misclassifications
- **False positives:** 1,216
  Predicted movement, but no movement occurred within 8 hours  

- **False negatives:** 765  
  Failed to predict actual movement within 8 hours  

#### Interpretation

The relatively low number of false negatives is important because the model captures most actual short-term movement events.

This supports the high recall observed earlier.

The model is therefore effective at identifying devices likely to become active soon, although some overprediction remains.

## 4.6 Additional prediction horizons

To evaluate how prediction performance changes across different planning windows, XGBoost is also tested for:

- movement within 4 hours  
- movement within 12 hours  

This allows comparison of short-term forecasting sensitivity across horizons.

In [ ]:
# Target: movement within next 4 hours
df["target_move_4h"] = (
    df["time_to_next_hours"] <= 4
).astype(int)

In [ ]:
# Target: movement within next 12 hours
df["target_move_12h"] = (
    df["time_to_next_hours"] <= 12
).astype(int)

#### XGBoost for 4-hour horizon

In [ ]:
# Prepare data
y_4h = df["target_move_4h"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_4h,
    test_size=0.2,
    random_state=42,
    stratify=y_4h
)

# Train model
xgb_model_4h = Pipeline([
    ("preprocessing", preprocessor),
    (
        "classifier",
        XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            random_state=42,
            eval_metric="logloss"
        )
    )
])

xgb_model_4h.fit(X_train, y_train)

# Predict
y_pred_4h = xgb_model_4h.predict(X_test)
y_prob_4h = xgb_model_4h.predict_proba(X_test)[:, 1]

# Evaluate
evaluate_model(
    y_test,
    y_pred_4h,
    y_prob_4h,
    "XGBoost - 4 hours"
)

### Compare multiple prediction horizons

To refine the operational definition of short-term utilization, XGBoost is compared across five alternative time windows:

- 4 hours  
- 6 hours  
- 8 hours  
- 10 hours  
- 12 hours

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from xgboost import XGBClassifier

# ---------------------------
# 1. Sort data (time-based)
# ---------------------------
df = df.sort_values("timestamp")

# ---------------------------
# 2. Split (60/20/20)
# ---------------------------
n = len(df)

train_end = int(n * 0.6)
val_end = int(n * 0.8)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

# ---------------------------
# 3. Horizons
# ---------------------------
horizons = [4, 6, 8, 10, 12]

# ---------------------------
# 4. Create ALL targets (before loop)
# ---------------------------
for h in horizons:
    df[f"target_move_{h}h"] = (df["time_to_next_hours"] <= h).astype(int)

# Assign targets to splits
for h in horizons:
    target_col = f"target_move_{h}h"
    
    train_df[target_col] = df.iloc[:train_end][target_col]
    val_df[target_col] = df.iloc[train_end:val_end][target_col]
    test_df[target_col] = df.iloc[val_end:][target_col]

# ---------------------------
# 5. Storage
# ---------------------------
results = []
models = {}

# ---------------------------
# 6. Loop over horizons
# ---------------------------
for h in horizons:
    
    target_col = f"target_move_{h}h"
    
    # Features and target
    X_train = train_df[features]
    y_train = train_df[target_col]
    
    X_val = val_df[features]
    y_val = val_df[target_col]
    
    X_test = test_df[features]
    y_test = test_df[target_col]
    
    # Model pipeline
    model = Pipeline([
        ("preprocessing", preprocessor),
        (
            "classifier",
            XGBClassifier(
                n_estimators=200,
                max_depth=6,
                learning_rate=0.1,
                random_state=42,
                eval_metric="logloss"
            )
        )
    ])
    
    # Train model
    model.fit(X_train, y_train)
    
    # Save model
    models[h] = model
    
    # Predictions
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    # Save results
    results.append({
        "horizon_hours": h,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob)
    })

# ---------------------------
# 7. Results dataframe
# ---------------------------
results_df = pd.DataFrame(results)
results_df_rounded = results_df.round(3)

print(results_df_rounded)

# ---------------------------
# 8. Save table as JPEG
# ---------------------------
fig, ax = plt.subplots()
ax.axis('tight')
ax.axis('off')

table = ax.table(
    cellText=results_df_rounded.values,
    colLabels=results_df_rounded.columns,
    loc='center'
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.auto_set_column_width(col=list(range(len(results_df_rounded.columns))))

plt.savefig("results_table.jpeg", bbox_inches='tight', dpi=300)
plt.close()

print("Table saved as 'results_table.jpeg'")

# ---------------------------
# 9. Save models to disk
# ---------------------------
import os
import joblib

model_dir = "../models"
os.makedirs(model_dir, exist_ok=True)

for h, model in models.items():
    filepath = os.path.join(model_dir, f"model_{h}h.pkl")
    joblib.dump(model, filepath)

print(f"Models saved to {model_dir}")


Model performance improves from 4 to 8 hours, where ROC-AUC reaches its highest value (0.785).  
Performance remains similar at 10 hours but declines slightly at 12 hours in terms of class discrimination.

Although longer horizons increase recall, prediction also becomes less strict because movement is more likely to occur eventually.

Overall, the **8-hour horizon provides the best balance between predictive performance and operational relevance**, and is therefore selected as the primary target definition.

## 4.7 Hyperparameter tuning

To further optimize predictive performance, hyperparameter tuning is applied to the XGBoost model.

A grid search is used to test alternative parameter combinations and identify the settings that maximize ROC-AUC.

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
# Use selected 8-hour target
y_train = train_df["target_move_8h"]
y_val = val_df["target_move_8h"]
y_test = test_df["target_move_8h"]

X_train = train_df[features]
X_val = val_df[features]
X_test = test_df[features]

# Base pipeline
xgb_tuning_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    (
        "classifier",
        XGBClassifier(
            random_state=42,
            eval_metric="logloss"
        )
    )
])

# Parameter grid
param_grid = {
    "classifier__n_estimators": [100, 200, 300, 500],
    "classifier__max_depth": [3, 4, 6, 8, 10],
    "classifier__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "classifier__subsample": [0.8, 1.0],
    "classifier__colsample_bytree": [0.8, 1.0]
}

In [ ]:
# Grid search (performed on training data only)
grid_search = GridSearchCV(
    estimator=xgb_tuning_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=3,
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

#### Best parameter settings

In [ ]:
print("Best parameters:")
print(grid_search.best_params_)

print("\nBest ROC-AUC:")
print(grid_search.best_score_)

In [ ]:
#### Evaluate tuned model on validation data

# Best tuned model
best_model = grid_search.best_estimator_

# Predictions
y_pred_tuned = best_model.predict(X_val)
y_prob_tuned = best_model.predict_proba(X_val)[:, 1]

# Evaluation
evaluate_model(
    y_val,
    y_pred_tuned,
    y_prob_tuned,
    "Tuned XGBoost"
)

#### Tuned model assessment

Hyperparameter tuning did not meaningfully improve performance compared with the original XGBoost model.

The results remain nearly identical in accuracy, F1-score, and ROC-AUC, suggesting that the initial parameter settings were already close to optimal for this dataset.

The original XGBoost model is therefore retained as the preferred final model.

## 4.8 Temporal feature engineering

To improve predictive performance, additional features are constructed to capture temporal dynamics in device behaviour.

These features incorporate information about previous movements and recent activity patterns.

Importantly, all features are based only on past information to avoid data leakage.

In [ ]:
# Create copy for temporal feature engineering
df_temp = df.copy()

# Ensure correct sorting BEFORE lag features
df_temp = df_temp.sort_values(["object_key", "timestamp"])

# Previous stay duration
df_temp["prev_stay_duration"] = (
    df_temp.groupby("object_key")["stay_duration_sec"]
    .shift(1)
)

# Previous movement duration
df_temp["prev_duration"] = (
    df_temp.groupby("object_key")["duration_sec"]
    .shift(1)
)

# Time since previous movement (in hours)
df_temp["time_since_prev"] = (
    df_temp["timestamp"] - df_temp.groupby("object_key")["timestamp"].shift(1)
).dt.total_seconds() / 3600

# Rolling movement count (last 3 events)
df_temp["movements_last_3"] = (
    df_temp.groupby("object_key")["object_key"]
    .rolling(3)
    .count()
    .reset_index(level=0, drop=True)
)

df_temp.head()

### Handling missing values

Lag features introduce missing values for the first observation of each device.

These are handled during preprocessing using imputation, ensuring compatibility with machine learning models.

### Update feature set

The feature set is extended to include temporal dynamics.

In [ ]:
features = [
    "distance_mtr",
    "duration_sec",
    "stay_duration_sec",
    "segment_count",
    "distance_floor",
    "hour",
    "time_period",
    "movement_ratio",
    "avg_speed",
    "activity_level",
    "sl_floor_local_name",
    
    # NEW temporal features
    "prev_stay_duration",
    "prev_duration",
    "time_since_prev",
    "movements_last_3"
]

X = df_temp[features]

### Split into train/val/test

In [ ]:
df_temp = df_temp.sort_values("timestamp")

n = len(df_temp)

train_end = int(n * 0.6)
val_end = int(n * 0.8)

train_df_temp = df_temp.iloc[:train_end]
val_df_temp = df_temp.iloc[train_end:val_end]
test_df_temp = df_temp.iloc[val_end:]

### Re-train model with temporal features

The XGBoost model is re-trained using the extended feature set to evaluate performance improvements.

In [ ]:
from xgboost import XGBClassifier

# Use selected 8-hour target
y_train = train_df_temp["target_move_8h"]
y_val = val_df_temp["target_move_8h"]
y_test = test_df_temp["target_move_8h"]

X_train = train_df_temp[features]
X_val = val_df_temp[features]
X_test = test_df_temp[features]

# Train model
xgb_model_updated = Pipeline([
    ("preprocessing", preprocessor),
    (
        "classifier",
        XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            random_state=42,
            eval_metric="logloss"
        )
    )
])

xgb_model_updated.fit(X_train, y_train)

# Predict (validation set)
y_pred_updated = xgb_model_updated.predict(X_val)
y_prob_updated = xgb_model_updated.predict_proba(X_val)[:, 1]

# Evaluate
evaluate_model(
    y_val,
    y_pred_updated,
    y_prob_updated,
    "XGBoost with temporal features"
)

## 4.9 Final model evaluation on test set

After model selection and validation, the final XGBoost model is evaluated on the held-out test set. 

The test set consists of the most recent observations and has not been used during model training or selection. This ensures an unbiased estimate of model performance on unseen future data.

This evaluation reflects the model’s ability to generalize to new observations in a realistic operational setting.

In [ ]:
# Final evaluation on test set

best_model = xgb_model  # eller tuned / updated

y_pred_test = best_model.predict(X_test)
y_prob_test = best_model.predict_proba(X_test)[:, 1]

evaluate_model(
    y_test,
    y_pred_test,
    y_prob_test,
    "Final model (test set)"
)

## Output 

In [ ]:
# Create output with predicted probabilities (test set)

output_df = test_df.copy()

# Add predicted probabilities
output_df["predicted_probability"] = y_prob_test

# Keep only relevant columns for interpretation
output_df = output_df[[
    "object_key",
    "timestamp",
    "predicted_probability"
]]

# Sort by highest probability (most likely to move)
output_df_sorted = output_df.sort_values(
    by="predicted_probability",
    ascending=False
)

# Show top 10 devices
output_df_sorted.head(10)

In [ ]:
# Aggregate to device level (max probability per device)
device_priority = (
    output_df
    .groupby("object_key")["predicted_probability"]
    .max()
    .reset_index()
)

# Sort by highest priority
device_priority = device_priority.sort_values(
    by="predicted_probability",
    ascending=False
)

# Show top devices
device_priority.tail(20)

## Treshold categorising

In [ ]:
# Define probability thresholds
def classify_probability(p):
    if p < 0.3:
        return "low"
    elif p < 0.7:
        return "medium"
    else:
        return "high"

# Apply classification
output_df["probability_group"] = output_df["predicted_probability"].apply(classify_probability)

In [ ]:
# Count devices in each group
group_counts = output_df["probability_group"].value_counts()

print(group_counts)

In [ ]:
# Aggregate to device level
device_priority = (
    output_df
    .groupby("object_key")["predicted_probability"]
    .max()
    .reset_index()
)

# Apply grouping
device_priority["probability_group"] = device_priority["predicted_probability"].apply(classify_probability)

# Distribution
device_priority["probability_group"].value_counts()

In [ ]:
# Merge device-level probability with activity level
merged = device_priority.merge(
    df[["object_key", "activity_level"]],
    on="object_key",
    how="left"
)

# Remove duplicates (one row per device)
merged = merged.drop_duplicates(subset="object_key")

# Crosstab
pd.crosstab(
    merged["activity_level"],
    merged["probability_group"],
    normalize="index"
)

### Consistency between historical activity and predicted probabilities

To assess whether the model captures meaningful patterns in equipment behavior, predicted probabilities are compared with device-level activity groups derived from the descriptive analysis.

The results show a clear relationship between historical activity and predicted short-term movement:

- High-activity devices are strongly associated with high predicted probabilities. Approximately 81% of these devices are classified as high probability, while only a very small share (≈2.5%) are assigned low probability.
- Low-activity devices exhibit more variation. While around 22% are classified as low probability, a substantial share is assigned medium (≈51%) or even high probability (≈28%).

These findings indicate that predicted probabilities are broadly consistent with historical activity patterns, confirming that the model captures persistent differences in device behavior.

At the same time, the variation observed among low-activity devices suggests that historical activity does not fully determine short-term movement. Some devices with low overall activity are still predicted to become active within the defined time window.

This highlights that the model incorporates additional temporal and contextual information beyond aggregate activity levels, allowing it to capture short-term dynamics in equipment behavior rather than simply reproducing historical patterns.

In [ ]:
df.groupby("activity_level")["distance_floor"].describe()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------
# 1. Merge device-level data
# ---------------------------
merged = device_priority.merge(
    df[["object_key", "activity_level"]],
    on="object_key",
    how="left"
)

# One row per device
merged = merged.drop_duplicates(subset="object_key")

# ---------------------------
# 2. Crosstab (proportions)
# ---------------------------
ct = pd.crosstab(
    merged["activity_level"],
    merged["probability_group"],
    normalize="index"
)

# Ensure logical order
ct = ct[["low", "medium", "high"]]

# ---------------------------
# 3. Plot
# ---------------------------
fig, ax = plt.subplots(figsize=(8, 5))

ct.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    colormap="viridis"
)

# Titles and labels
ax.set_title("Relationship Between Device Activity and Predicted Movement Probability")
ax.set_xlabel("Activity Level")
ax.set_ylabel("Proportion")

# Legend
ax.legend(title="Probability Group")

# Rotate x labels
plt.xticks(rotation=0)

# ---------------------------
# 4. Add percentage labels
# ---------------------------
for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.2f",
        label_type="center"
    )

# ---------------------------
# 5. Save figure
# ---------------------------
plt.tight_layout()
plt.savefig("activity_vs_probability.png", dpi=300)

plt.show()